# Chisel: train on Colab

Trains the emoji diffusion model on a free Colab GPU and hands back a zip with the checkpoint, sample grids, evaluation numbers and the ONNX file for the web app.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all. The base model takes roughly 30 to 60 minutes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "No GPU: set Runtime > Change runtime type > T4 GPU"

## 1. Get the code

In [ ]:
BRANCH = "dev"
!rm -rf Chisel && git clone --depth 1 -b $BRANCH https://github.com/Cheeriooo/Chisel.git
%cd Chisel
!pip -q install onnx onnxruntime onnxscript

## 2. (Optional) keep results on Google Drive

Colab deletes everything when the session ends. Run this cell to store `runs/` on your Drive, so a disconnect loses nothing and re-running the training cell resumes from the last checkpoint. Skip it if you would rather just download the zip at the end.

In [ ]:
USE_DRIVE = False  # set to True, then run this cell
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/chisel_runs && ln -sfn /content/drive/MyDrive/chisel_runs runs

## 3. Build the dataset

Downloads three emoji sets (about 190 MB), converts them to 32×32 and prints the per-class counts. Expect 4,548 images, 4,096 of them for training.

In [ ]:
!python train/data.py
from IPython.display import Image, display
display(Image("runs/data_grid.png", width=900))

## 4. Train

`--amp` turns on mixed precision, which the T4 handles well. A sample grid (one row per class) is saved every 2,500 steps to `runs/base/`. Re-running this cell resumes from the last checkpoint.

In [ ]:
!python train/train.py --name base --pred v --steps 40000 --amp

In [ ]:
import glob
latest = sorted(glob.glob("runs/base/samples_*.png"))[-1]
print(latest); display(Image(latest, width=560))

## 5. Evaluate

Trains the real-data classifier (its held-out accuracy is the ceiling), then the step and guidance ablations and the memorization check.

In [ ]:
!python train/eval.py classifier
!python train/eval.py ablate --run base
!python train/eval.py memorize --run base
for name in ["ablation_steps", "ablation_guidance", "memorization"]:
    print(name); display(Image(f"runs/base/{name}.png", width=900))

## 6. Sketch and inpainting grids, GIF

In [ ]:
for what in ["grid", "sdedit", "inpaint", "gif"]:
    !python train/sample.py $what --run base
for name in ["grid", "sdedit", "inpaint"]:
    print(name); display(Image(f"runs/base/{name}.png", width=700))

## 7. Export to ONNX

Writes `runs/base/export/`: the model, the schedule table and the parity fixtures the web tests use.

In [ ]:
!python export/export_onnx.py --run base

## 8. Download

Unzip into the repo's `runs/` folder on your machine, so that `runs/base/ckpt.pt` exists.

In [ ]:
!cd runs && zip -q -r ../chisel_base.zip base classifier.pt
from google.colab import files
files.download("chisel_base.zip")

## 9. (Optional) model-size ablation

Two more runs, about 0.9M and 8.4M parameters, to see where quality stops improving. The large one takes a few times longer than the base run.

In [ ]:
RUN_ABLATION = False  # set to True, then run this cell
if RUN_ABLATION:
    !python train/train.py --name small --ch 16 --pred v --steps 40000 --amp
    !python train/train.py --name large --ch 48 --pred v --steps 40000 --amp
    for run in ["small", "large"]:
        !python train/eval.py ablate --run $run
        !python train/eval.py memorize --run $run
        !python export/export_onnx.py --run $run
    !cd runs && zip -q -r ../chisel_sizes.zip small large
    files.download("chisel_sizes.zip")